<a href="https://colab.research.google.com/github/aminsiddik2810/data-science-2026/blob/https%2Fgithub.com%2Faminsiddik2810%2Fdata-science-2026.git/Pertemuan_10_AminSiddikRangkuti_220401010124.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 10 — Algoritma Klasifikasi Bagian 2

**Nama:** Amin Siddik Rangkuti  
**NIM:** 220401010124  
**Program Studi:** PJJ Informatika  
**Mata Kuliah:** Pengantar Data Science  

## Topik
Notebook ini membahas praktik klasifikasi menggunakan **Random Forest** untuk prediksi **Customer Churn**. Materi utama meliputi Ensemble Learning, Bagging, Random Forest, Imbalanced Dataset, evaluasi klasifikasi, dan feature importance.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aminsiddik2810/data-science-2026/blob/main/Pertemuan_10_AminSiddikRangkuti_220401010124.ipynb)

## 1. Tujuan Praktikum

Setelah menyelesaikan notebook ini, mahasiswa diharapkan mampu:

1. Memahami konsep **Ensemble Learning**.
2. Menjelaskan cara kerja **Random Forest**.
3. Membangun model klasifikasi untuk prediksi customer churn.
4. Mengevaluasi model menggunakan accuracy, precision, recall, F1-score, ROC-AUC, dan confusion matrix.
5. Menangani masalah **imbalanced dataset** menggunakan `class_weight="balanced"`.
6. Menampilkan **feature importance** untuk melihat fitur yang paling berpengaruh.

## 2. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay
)

import warnings
warnings.filterwarnings("ignore")

print("Library berhasil di-import")

## 3. Membuat Dataset Customer Churn

Dataset dibuat secara sintetis agar notebook dapat langsung dijalankan tanpa perlu file eksternal. Target `Churn` menunjukkan apakah pelanggan berhenti menggunakan layanan atau tidak.

In [ ]:
np.random.seed(42)

n_samples = 600

tenure = np.random.randint(1, 73, n_samples)
monthly_charges = np.round(np.random.uniform(20, 120, n_samples), 2)
total_charges = np.round(tenure * monthly_charges + np.random.normal(0, 250, n_samples), 2)
total_charges = np.where(total_charges < 0, monthly_charges, total_charges)

contract = np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.55, 0.25, 0.20])
internet_service = np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.38, 0.45, 0.17])
payment_method = np.random.choice(
    ['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'],
    n_samples, p=[0.40, 0.20, 0.20, 0.20]
)

# Membuat probabilitas churn berdasarkan pola umum
prob_churn = (
    0.10
    + (contract == 'Month-to-month') * 0.25
    + (internet_service == 'Fiber optic') * 0.12
    + (payment_method == 'Electronic check') * 0.10
    + (tenure < 12) * 0.18
    + (monthly_charges > 85) * 0.10
)
prob_churn = np.clip(prob_churn, 0, 0.85)

churn = np.random.binomial(1, prob_churn)

df = pd.DataFrame({
    'tenure': tenure,
    'MonthlyCharges': monthly_charges,
    'TotalCharges': total_charges,
    'Contract': contract,
    'InternetService': internet_service,
    'PaymentMethod': payment_method,
    'Churn': churn
})

df.head()

## 4. Informasi Dataset

In [ ]:
print("Jumlah baris dan kolom:", df.shape)
print("\nInformasi dataset:")
df.info()

print("\nJumlah nilai kosong:")
print(df.isnull().sum())

## 5. Distribusi Target Churn

Pada kasus dunia nyata, data churn sering tidak seimbang. Oleh karena itu, distribusi kelas perlu dicek sebelum model dibuat.

In [ ]:
target_count = df['Churn'].value_counts().sort_index()
print(target_count)
print("\nPersentase target:")
print((target_count / len(df) * 100).round(2))

target_count.plot(kind='bar')
plt.title('Distribusi Target Churn')
plt.xlabel('Churn (0 = Tidak, 1 = Ya)')
plt.ylabel('Jumlah Data')
plt.xticks(rotation=0)
plt.show()

## 6. Memisahkan Fitur dan Target

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = ['Contract', 'InternetService', 'PaymentMethod']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Data train:", X_train.shape)
print("Data test :", X_test.shape)

## 7. Preprocessing Data

Fitur numerik dinormalisasi menggunakan `StandardScaler`, sedangkan fitur kategorikal diubah menjadi bentuk numerik menggunakan `OneHotEncoder`.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

print("Preprocessor siap digunakan")

## 8. Membuat Model Random Forest

Random Forest termasuk metode **bagging**, yaitu membangun banyak pohon keputusan secara paralel dan menggabungkan hasil prediksinya. Parameter `class_weight="balanced"` digunakan untuk membantu menangani data yang tidak seimbang.

In [ ]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)
print("Model Random Forest berhasil dilatih")

## 9. Prediksi Model

In [ ]:
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

result = X_test.copy()
result['Actual_Churn'] = y_test.values
result['Predicted_Churn'] = y_pred
result['Probability_Churn'] = np.round(y_proba, 3)

result.head(10)

## 10. Evaluasi Model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

evaluation = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Score': [accuracy, precision, recall, f1, roc_auc]
})

evaluation['Score'] = evaluation['Score'].round(4)
evaluation

## 11. Classification Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Tidak Churn', 'Churn']))

## 12. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.xticks([0, 1], ['Tidak Churn', 'Churn'])
plt.yticks([0, 1], ['Tidak Churn', 'Churn'])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center')

plt.colorbar()
plt.show()

## 13. ROC Curve

In [ ]:
RocCurveDisplay.from_estimator(rf_model, X_test, y_test)
plt.title('ROC Curve - Random Forest')
plt.show()

## 14. Feature Importance

Feature importance digunakan untuk melihat fitur mana yang paling berpengaruh dalam prediksi churn.

In [ ]:
trained_preprocessor = rf_model.named_steps['preprocessor']
trained_classifier = rf_model.named_steps['classifier']

cat_names = trained_preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
feature_names = numeric_features + list(cat_names)

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': trained_classifier.feature_importances_
}).sort_values(by='Importance', ascending=False)

importance_df.head(10)

In [ ]:
top_features = importance_df.head(10).sort_values('Importance')

plt.figure(figsize=(8, 5))
plt.barh(top_features['Feature'], top_features['Importance'])
plt.title('Top 10 Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

## 15. Prediksi Data Pelanggan Baru

In [ ]:
new_customer = pd.DataFrame({
    'tenure': [5],
    'MonthlyCharges': [95.0],
    'TotalCharges': [475.0],
    'Contract': ['Month-to-month'],
    'InternetService': ['Fiber optic'],
    'PaymentMethod': ['Electronic check']
})

pred_label = rf_model.predict(new_customer)[0]
pred_proba = rf_model.predict_proba(new_customer)[:, 1][0]

print("Prediksi Churn:", "Ya" if pred_label == 1 else "Tidak")
print("Probabilitas Churn:", round(pred_proba, 3))

## 16. Kesimpulan

Berdasarkan praktikum ini, dapat disimpulkan bahwa:

1. **Random Forest** merupakan algoritma ensemble berbasis banyak decision tree.
2. Random Forest menggunakan bootstrap sampling dan pemilihan fitur acak agar pohon lebih beragam.
3. Model ini cocok untuk klasifikasi customer churn karena mampu menangani pola non-linear.
4. Pada data tidak seimbang, akurasi saja tidak cukup; perlu melihat precision, recall, F1-score, dan ROC-AUC.
5. Parameter `class_weight="balanced"` dapat membantu model memperhatikan kelas minoritas.
6. Feature importance membantu mengetahui faktor utama yang memengaruhi pelanggan berhenti atau tetap menggunakan layanan.